## Парсинг английских текстов 
#### + выгрузка названий песен из либретто на русском

In [ ]:
# импорты

import re
import os
import pandas as pd

from tqdm import tqdm


Извлечем названия песен из файлов либретто на русском, чтобы удобно сопоставить их с английскими

In [3]:
path = 'c:\\Users\\Butak\\Desktop\\Комп. лингвистика\\Проект-2\\libretti'
os_content = os.listdir(path)

In [4]:
filenames = [el for el in os_content if el.endswith('.txt')]
print(filenames)

['cats_rus.txt', 'chaplin_rus.txt', 'chess_rus.txt', 'hairs_rus.txt', 'jekill_hyde_rus.txt', 'jesus_rus_mossovet.txt', 'mammamia_rus.txt']


In [5]:
texts = []
for i in filenames:
    with open(f'{path}/{i}', encoding = 'utf-8') as file:
        text = file.read()
        title = re.search(r'(?<=---\n).+\n', text).group().strip()
        texts.append((title, text))

In [6]:
df_texts = pd.DataFrame(texts, columns = ['Название', 'Текст'])
df_texts

,Название,Текст
0,Кошки,---\nКошки\nCats\nПеревод: Алексей Кортнев\n--...
1,Чаплин,---\nЧаплин\nChaplin\nПеревод: Константин Руби...
2,Шахматы,---\nШахматы\nChess\nПеревод: Алексей Иващенко...
3,Волосы,----------------------------------------------...
4,Джекилл и Хайд,---\nДжекилл и Хайд\nТеатр музыкальной комедии...
5,Иисус Христос Суперзвезда (Театр им. Моссовета),---\nИисус Христос Суперзвезда (Театр им. Мосс...
6,MAMMA MIA!,---\nMAMMA MIA!\n---\n\n\n\nАКТ 1\n\n\n\n1. МИ...


In [ ]:
def get_songs(row_num):
    row_num = int(row_num)
    row = df_texts.iloc[row_num, :]
    
    name = row.loc['Название']
    text = row.loc['Текст']
    songtitles = [re.sub(r'\s+', ' ', i).strip() for i in re.findall(r'\n+\d{1,2}[\.\)].+?\n+', text)]
    to_write = '\n'.join(songtitles)
    to_write = re.sub(r'\d{1,2}\.\s', '', to_write)
    with open(f'{path}\\titles\\{name}.txt', 'w', encoding = 'utf-8') as file:
        file.write(to_write) 

    return 'Success!'



In [ ]:
get_songs(1)

Название                                              Шахматы
Текст       ---\nШахматы\nChess\nПеревод: Алексей Иващенко...
Name: 2, dtype: object

# ПАРСИНГ

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
import re

In [41]:
url = 'https://www.allmusicals.com/m/mammamia.htm'
page = requests.get(url)
soup = BeautifulSoup(page.text)

In [10]:
baselink = 'https://www.allmusicals.com/'

In [11]:
songlinks = [baselink + link.get('href') for link in soup.find('section', {'class':'lyrics-list'}).find_all('a')]
titles_local = []
texts_local = []

In [ ]:
page = requests.get(songlinks[3])
soup = BeautifulSoup(page.text)

songtitle = soup.find('h2', {'class':'visible-print'}).text
titles_local.append(songtitle)

page_div = soup.find('div', id='page')  
if page_div:
    # 1й элемент - заголовок, последний - дата обновления, их в баню
    lines = [s.strip('"').strip() for s in page_div.stripped_strings][1:-1]
    songtext = '\n'.join(lines)
    texts_local.append(songtext)


In [ ]:
def preprocessing(text):
    edited = text.lower()
    edited = re.sub(r'\s+', ' ', edited) # замена всех пробельных символов на обычный пробел
    edited = re.sub(r' - ', '', edited) # убираем дефисы, которые на самом деле не дефисы (не внутри слова)
    edited = re.sub(r'[^А-ЯЁа-яё -]', '', edited) # удаление всего, что не кириллица и не дефис и пробел

    tokenized = word_tokenize(edited) # токенизация по словам

    lemmatized = [morph.parse(token)[0].normal_form for token in tokenized] # лемматизация

    tokens_clean = [token for token in lemmatized if token not in stopwords_custom] # очистка от стоп-слов

    return(tokens_clean)

In [ ]:
def songs_from_web(url):
    # страница треков мюзикла
    page_main = requests.get(url)
    soup_main = BeautifulSoup(page_main.text)
    musical_main = re.sub(r' Lyrics.+', '' , soup_main.find('h2').text, flags=re.DOTALL)
    if not musical_main:
        musical_main = 'NAME NOT FOUND'

    # получение ссылок на песни
    songlinks = [baselink + link.get('href') for link in soup_main.find('section', {'class':'lyrics-list'}).find_all('a')]

    # для хранения названий и текстов
    titles_local = []
    texts_local = []    

    # сбор названий и текстов по ссылкам
    for link in songlinks:
        page = requests.get(link)
        soup = BeautifulSoup(page.text)

        songtitle = soup.find('h2', {'class':'visible-print'}).text
        titles_local.append(songtitle)

        page_div = soup.find('div', id='page')  
        if page_div:
            # 1й элемент - заголовок, последний - дата обновления, их в баню
            lines = [s.strip('"').strip() for s in page_div.stripped_strings][1:-1]
            songtext = '\n'.join(lines)
            texts_local.append(songtext)

    check = True if len(titles_local) == len(texts_local) else False
    n = len(titles_local)
    data = [[musical_main]*n, titles_local, texts_local]

    return check, data

In [155]:
musicals_links = ['https://www.allmusicals.com/c/cats.htm',
'https://www.allmusicals.com/m/mammamia.htm',
'https://www.allmusicals.com/c/chess.htm',
'https://www.allmusicals.com/l/littlemermaid.htm',
'https://www.allmusicals.com/t/ticktickboom.htm',
'https://www.allmusicals.com/j/jesuschristsuperstar.htm',
'https://www.allmusicals.com/s/soundofmusicthe.htm',
'https://www.allmusicals.com/l/littleshopofhorrors.htm',
'https://www.allmusicals.com/p/phantomoftheoperathe.htm',
'https://www.allmusicals.com/c/chaplin.htm',
'https://www.allmusicals.com/c/chicago.htm',
'https://www.allmusicals.com/d/disneysbeautyandthebeast.htm',
'https://www.allmusicals.com/g/ghostthemusical.htm',
'https://www.allmusicals.com/j/jekyllhyde.htm',
'https://www.allmusicals.com/m/misssaigon.htm',
'https://www.allmusicals.com/c/cabaret.htm',
'https://www.allmusicals.com/h/hair.htm',
'https://www.allmusicals.com/d/deathnote.htm',
'https://www.allmusicals.com/s/singinintherain.htm',
'https://www.allmusicals.com/n/notredamedeparis.htm']


In [121]:
songs_data = pd.DataFrame(columns = ['musical', 'titles_global', 'texts_global'])
songs_data

,musical,titles_global,texts_global


In [110]:
check, test = songs_from_web(musicals_links[3])

In [114]:
print(len(test[0]))
print(len(test[1]))
print(len(test[2]))

29
29
29


In [133]:
musicals_links[4:]

['https://www.allmusicals.com/t/ticktickboom.htm',
 'https://www.allmusicals.com/j/jesuschristsuperstar.htm',
 'https://www.allmusicals.com/s/soundofmusicthe.htm',
 'https://www.allmusicals.com/l/littleshopofhorrors.htm',
 'https://www.allmusicals.com/p/phantomoftheoperathe.htm',
 'https://www.allmusicals.com/c/chaplin.htm',
 'https://www.allmusicals.com/c/chicago.htm',
 'https://www.allmusicals.com/d/disneysbeautyandthebeast.htm',
 'https://www.allmusicals.com/g/ghostthemusical.htm',
 'https://www.allmusicals.com/z/zorro.htm',
 'https://www.allmusicals.com/f/firstdate.htm',
 'https://www.allmusicals.com/j/jekyllhyde.htm',
 'https://www.allmusicals.com/m/misssaigon.htm',
 'https://www.allmusicals.com/c/cabaret.htm',
 'https://www.allmusicals.com/h/hair.htm',
 'https://www.allmusicals.com/d/deathnote.htmhttps://www.allmusicals.com/s/singinintherain.htm',
 'https://www.allmusicals.com/n/notredamedeparis.htm']

In [ ]:
musicals_links[1]

'https://www.allmusicals.com/d/deathnote.htm'

In [ ]:
# на всякий случай, если вдруг не совпадут кол-во текстов и кол-во названий
empty_rows = pd.DataFrame(index=range(3), columns=songs_data.columns)

for musical in musicals_links[19]:
    check, data_lyrics = songs_from_web(musical)

    df_local = pd.DataFrame({
    'musical': data_lyrics[0],
    'titles_global': data_lyrics[1],
    'texts_global': data_lyrics[2]
})

    songs_data = pd.concat([songs_data, df_local], ignore_index=True)
    if check == True:
        print(data_lyrics[0][0], ' - ок')
        pass
    else:
        print(data_lyrics[0][0], ' - ОШИБКА')
        songs_data = pd.concat([songs_data, empty_rows], ignore_index=True)
    

In [ ]:
# songs_data.to_excel('eng_songs.xlsx', index = False)